In [ ]:
def read_smiles_from_txt(filepath: str):

    smiles_list = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            
            smi = line.strip()
            
            if smi:
                smiles_list.append(smi)
                
    return smiles_list

# Validity, Novelty, Uniqueness

In [ ]:
from rdkit import Chem
nsample = 10000

origin_smiles = read_smiles_from_txt("../data.txt")
all_smiles = []
for i in origin_smiles:
    m = Chem.MolFromSmiles(i)
    smi = Chem.MolToSmiles(m)
    all_smiles.append(smi)

valid_entries = read_smiles_from_txt("emigen_mols.txt")
valid_smiles = []
for i in valid_entries:
    m = Chem.MolFromSmiles(i)
    smi = Chem.MolToSmiles(m)
    valid_smiles.append(smi)

validaty = round(len(valid_smiles)/nsample , 4)

novel_entries = [i for i in valid_smiles if i not in all_smiles]
novelty = round(len(novel_entries)/len(valid_smiles), 4)

unique_smiles = list(set(valid_smiles))

uniqueness = round(len(unique_smiles)/len(valid_smiles), 4)

print(f'Sample data {len(nsample)},  {novelty} novelty, {validaty} validity, {uniqueness} uniqueness')

# Plausibiilty

In [ ]:
from __future__ import annotations

from copy import deepcopy
from typing import Any

import pandas as pd
from rdkit import RDLogger
from rdkit.Chem.inchi import InchiReadWriteError
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.rdmolops import DetectChemistryProblems, GetMolFrags, SanitizeFlags, SanitizeMol

import logging
import re
from copy import deepcopy
from re import findall

from rdkit.Chem.inchi import MolFromInchi, MolToInchi
from rdkit.Chem.MolStandardize.rdMolStandardize import Uncharger
from rdkit.Chem.rdchem import AtomValenceException, Mol
from rdkit.Chem.rdmolops import AssignStereochemistryFrom3D, RemoveHs, RemoveStereochemistry, SanitizeMol

import logging
import os
import sys
from typing import Any

import rdkit
from rdkit import rdBase
from copy import deepcopy
from logging import getLogger

import numpy as np
from rdkit import RDLogger
from rdkit.Chem.AllChem import AssignBondOrdersFromTemplate
from rdkit.Chem.Lipinski import HAcceptorSmarts, HDonorSmarts
from rdkit.Chem.rdchem import AtomValenceException, Bond, Conformer, GetPeriodicTable, Mol, RWMol
from rdkit.Chem.rdForceFieldHelpers import UFFGetMoleculeForceField
from rdkit.Chem.rdMolAlign import GetBestAlignmentTransform
from rdkit.Chem.rdmolfiles import MolFromSmarts
from rdkit.Chem.rdmolops import AddHs, RemoveHs, RemoveStereochemistry, RenumberAtoms, SanitizeMol
from rdkit.Chem.rdMolTransforms import TransformConformer
# redirect logs to Python logger
rdBase.LogToPythonLogger()

logger = logging.getLogger(__name__)

# https://github.com/rdkit/rdkit/discussions/5435
class CaptureLogger(logging.Handler):
    """Helper class that captures Python logger output."""

    def __init__(self, level: str | None = None) -> None:
        """Initialize logger."""
        super().__init__(level=logging.NOTSET if level is None else level)
        self.logs: dict[str, str] = {}
        self.devnull = open(os.devnull, "w")
        rdkit.log_handler.setStream(self.devnull)
        rdkit.logger.addHandler(self)

    def __enter__(self) -> dict[str, str]:
        """Enter context manager."""
        return self.logs

    def __exit__(self, exc_type: type[BaseException] | None, exc_val: BaseException | None, exc_tb: Any) -> None:
        """Exit context manager."""
        self.release()

    def handle(self, record: logging.LogRecord) -> bool:
        """Handle log record."""
        key = record.levelname
        val = self.format(record)
        self.logs[key] = self.logs.get(key, "") + val
        return False

    def release(self) -> None:
        """Release logger."""
        rdkit.log_handler.setStream(sys.stderr)
        rdkit.logger.removeHandler(self)
        self.devnull.close()

    def emit(self, record: logging.LogRecord) -> None:
        """Emit a log record."""
        
def add_stereo_hydrogens(mol: Mol) -> Mol:
    """Add all hydrogens but those on primary ketimine."""
    exclude = {match[1] for match in mol.GetSubstructMatches(MolFromSmarts("[CX3]=[NH1]"))}
    atoms = [a.GetIdx() for a in mol.GetAtoms() if a.GetAtomicNum() != 1 if a.GetIdx() not in exclude]
    mol = AddHs(mol, onlyOnAtoms=atoms, addCoords=True)
    return mol

def remove_isotopic_info(mol: Mol) -> Mol:
    """Remove isotopic information from molecule."""
    for atom in mol.GetAtoms():
        atom.SetIsotope(0)
    return mol

def neutralize_atoms(mol: Mol) -> Mol:
    """Add and remove hydrogens to neutralize charges ignoring overall charge."""
    # https://www.rdkit.org/docs/Cookbook.html#neutralizing-charged-molecules
    # stronger than rdkit.Chem.MolStandardize.rdMolStandardize.Uncharger
    try:
        pattern = MolFromSmarts("[+1!h0!$([*]~[-1,-2,-3,-4]),-1!$([*]~[+1,+2,+3,+4])]")
        at_matches = mol.GetSubstructMatches(pattern)
        at_matches_list = [y[0] for y in at_matches]
        if len(at_matches_list) > 0:
            for at_idx in at_matches_list:
                atom = mol.GetAtomWithIdx(at_idx)
                chg = atom.GetFormalCharge()
                hcount = atom.GetTotalNumHs()
                atom.SetFormalCharge(0)
                atom.SetNumExplicitHs(hcount - chg)
                atom.UpdatePropertyCache()
    except AtomValenceException:
        logger.warning("AtomValenceException raised while neutralizing molecule. Continuing with original molecule.")
        return mol
    return mol

def get_inchi(mol: Mol, inchi_strict: bool = False) -> str:
    """Get inchi of a molecule."""

    with CaptureLogger() as log:
        SanitizeMol(mol)
        inchi = str(MolToInchi(mol, treatWarningAsError=inchi_strict))
        # check inchi because inchi generation does not raise an error if the inchi is invalid
        if MolFromInchi(inchi, sanitize=True) is None:
            warnings = re.split(r"\[.+?\] WARNING: ", log.get("WARNING", ""))[1:]
            errors = re.split(r"\[.+?\] ERROR: ", log.get("ERROR", ""))[1:]
            message = f"InChI ERRORs: {'; '.join(errors)} WARNINGs: {'; '.join(warnings)}"
            raise Exception(message)
    return inchi


def standardize_and_get_inchi(mol: Mol, options: str = "", log_level=None, warnings_as_errors=False) -> str:
    """Return InChI after standardising molecule and inferring stereo from coordinates."""

    mol = deepcopy(mol)

    flags = SanitizeMol(mol, catchErrors=True)


    if flags:
        logger.debug("Cannot get InChI because molecule doesn't sanitize due to %s.", flags)
        return ""  

    # standardise molecule
    mol = remove_isotopic_info(mol)

    # assign stereo from 3D coordinates only if 3D coordinates are present
    has_pose = mol.GetNumConformers() > 0
    if has_pose:
        RemoveStereochemistry(mol)

    mol = RemoveHs(mol)
    try:
        mol = neutralize_atoms(mol)
    except AtomValenceException:
        logger.warning("Failed to neutralize molecule. Using uncharger. InChI check might fail.")
        mol = Uncharger().uncharge(mol)
    mol = add_stereo_hydrogens(mol)

    if has_pose:
        AssignStereochemistryFrom3D(mol, replaceExistingTags=True)

    with CaptureLogger():
        inchi = str(MolToInchi(mol, options=options, logLevel=log_level, treatWarningAsError=warnings_as_errors))

    return inchi


def is_valid_inchi(inchi: str) -> bool:
    """Check that InChI can be parsed and sanitization does not fail."""
    try:
        mol = MolFromInchi(inchi)
        assert mol is not None, "Molecule is None."
        assert not SanitizeMol(mol, catchErrors=True), "Molecule does not sanitize."
        return True
    except Exception:
        return False


def split_inchi(inchi: str) -> dict[str, str]:
    """Split the standard InChI string without isotopic info into its layers."""
    if not inchi.startswith("InChI="):
        raise ValueError("InChI string must start with 'InChI='")

    # inchi always InChi=1S/...formula.../...layer.../...layer.../...layer...
    version = inchi[6:].split(r"/", 1)[0]
    layers = findall(r"(?=.*)(\/[a-z]{0,1})(.*?)(?=\/.*|$)", inchi[6:])

    inchi_parts = {"=": version}
    for prefix, layer in layers:
        # standard inchi strings without isotopic info have each layer no more than once
        assert prefix not in inchi_parts, f"Layer {prefix} more than once!"
        inchi_parts[prefix] = layer

    return inchi_parts

def check_chemistry_using_rdkit(mol_pred: Mol) -> dict[str, dict[str, bool] | dict[str, bool | str] | pd.DataFrame]:
    """Check sanity of molecule using RDKit sanitization rules."""
    assert isinstance(mol_pred, Mol)
    mol = deepcopy(mol_pred)

    RDLogger.DisableLog("rdApp.*")  # type: ignore[attr-defined]
    errors = DetectChemistryProblems(mol, sanitizeOps=SanitizeFlags.SANITIZE_ALL)
    RDLogger.EnableLog("rdApp.*")  # type: ignore[attr-defined]

    messages = [error.Message() for error in errors]
    atom_indices = [e.GetAtomIndices() if hasattr(e, "GetAtomIndices") else e.GetAtomIdx() for e in errors]
    types = [error.GetType() for error in errors]

    details = pd.DataFrame(dict(messages=messages, atom_indices=atom_indices, types=types))

    results = {
        "passes_valence_checks": "AtomValenceException" not in types,
        "passes_kekulization": "AtomKekulizeException" not in types,
        "passes_rdkit_sanity_checks": len(errors) == 0,
    }
    return {"results": results, "details": details}


def check_chemistry_using_inchi(mol_pred: Mol) -> dict[str, dict[str, str | bool | None]]:
    """Check sanity of a molecule using InChI rules."""
    assert isinstance(mol_pred, Mol)
    mol = deepcopy(mol_pred)

    passes = False

    try:
        inchi = get_inchi(mol, inchi_strict=False)
        passes = True
        errors = ""
    except InchiReadWriteError as e:
        errors = str(e.args[1])
        inchi = None
    except Exception as e:
        errors = str(e)
        inchi = None

    return {"results": {"inchi_convertible": passes}, "details": dict(errors=errors, inchi=inchi)}


def check_all_atoms_connected(mol_pred: Mol) -> dict[str, dict[str, bool] | dict[str, int]]:
    """Check if all atoms in a molecule are connected."""
    assert isinstance(mol_pred, Mol)
    mol = deepcopy(mol_pred)

    num_frags = len(GetMolFrags(mol, asMols=False, sanitizeFrags=False))
    results = {"all_atoms_connected": num_frags <= 1}
    details = {"number_fragments": num_frags}

    return {"results": results, "details": details}


def check_chemistry(mol_pred: Mol) -> dict[str, Any]:
    """Check sanity using RDKit and connectedness.

    Notes:
    - Only for backward compatibility. Use `check_chemistry_using_rdkit` and `check_all_atoms_connected` separately.
    """

    rdkit_sanity = check_chemistry_using_rdkit(mol_pred)
    connectedness = check_all_atoms_connected(mol_pred)
    return {
        "results": rdkit_sanity["results"] | connectedness["results"],
        "details": rdkit_sanity["details"],
    }


def check_radicals(mol_pred: Mol) -> dict[str, Any]:
    """Check that a molecule has no radicals."""

    radicals_before_sanitization = any(a.GetNumRadicalElectrons() for a in mol_pred.GetAtoms())
    SanitizeMol(mol_pred, catchErrors=True)
    radicals_after_sanitization = any(a.GetNumRadicalElectrons() for a in mol_pred.GetAtoms())

    return {
        "results": {
            "no_radicals": not radicals_after_sanitization,
            "no_radicals_before_sanitization": not radicals_before_sanitization,
        }
    } 

# Detecting implausible molecules based on the method of PostBusters

In [ ]:

from __future__ import annotations

from collections.abc import Iterable
from copy import deepcopy
from logging import getLogger
from typing import Any

import numpy as np
import pandas as pd
from rdkit.Chem import MolFromSmarts
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.rdDistGeom import GetMoleculeBoundsMatrix
from rdkit.Chem.rdmolops import SanitizeMol



logger = getLogger(__name__)

col_lb = "lower_bound"
col_ub = "upper_bound"
col_pe = "percent_error"
col_bpe = "bound_percent_error"
col_bape = "bound_absolute_percent_error"

bound_matrix_params = {
    "set15bounds": True,
    "scaleVDW": True,
    "doTriangleSmoothing": True,
    "useMacrocycle14config": False,
}

col_n_bonds = "number_bonds"
col_shortest_bond = "shortest_bond_relative_length"
col_longest_bond = "longest_bond_relative_length"
col_n_short_bonds = "number_short_outlier_bonds"
col_n_long_bonds = "number_long_outlier_bonds"
col_n_good_bonds = "number_valid_bonds"
col_bonds_result = "bond_lengths_within_bounds"
col_n_angles = "number_angles"
col_extremest_angle = "most_extreme_relative_angle"
col_n_bad_angles = "number_outlier_angles"
col_n_good_angles = "number_valid_angles"
col_angles_result = "bond_angles_within_bounds"
col_n_noncov = "number_noncov_pairs"
col_closest_noncov = "shortest_noncovalent_relative_distance"
col_n_clashes = "number_clashes"
col_n_good_noncov = "number_valid_noncov_pairs"
col_clash_result = "no_internal_clash"

_empty_results = {
    col_n_bonds: np.nan,
    col_shortest_bond: np.nan,
    col_longest_bond: np.nan,
    col_n_short_bonds: np.nan,
    col_n_long_bonds: np.nan,
    col_bonds_result: np.nan,
    col_n_angles: np.nan,
    col_extremest_angle: np.nan,
    col_n_bad_angles: np.nan,
    col_angles_result: np.nan,
    col_n_noncov: np.nan,
    col_closest_noncov: np.nan,
    col_n_clashes: np.nan,
    col_clash_result: np.nan,
}

def bpe(val: float, lb: float, ub: float) -> float:
    """Calculate out of bounds percentage error."""
    if val < lb:
        return pe(val, lb)
    if val > ub:
        return pe(val, ub)
    return 0.0


def bape(val: float, lb: float, ub: float) -> float:
    """Calculate out of bounds absolute percentage error."""
    if val < lb:
        return ape(val, lb)
    if val > ub:
        return ape(val, ub)
    return 0.0


def pe(val_pred: float, val_true: float) -> float:
    """Calculate percentage error."""
    return (val_pred - val_true) / val_true



def symmetrize_conjugated_terminal_bonds(df: pd.DataFrame, mol: Mol) -> pd.DataFrame:
    """
    Symmetrize the lower and upper bounds of the conjugated terminal bonds so that
    the new lower and upper bounds are the minimum and maximum of the original
    lower and upper bounds for each pair of atom elements.

    Args:
        df: Dataframe with the bond geometry information and bounds.
        mol: RDKit molecule object (conformer id doesn't matter I think)

    Returns:
        Dataframe with the bond geometry information and bounds, lower/upper bounds
        for conjugated terminal bonds are symmetrized.
    """

    def _sort_bond_ids(bond_ids: tuple[tuple | list]) -> tuple[tuple, ...]:
        return tuple(tuple(sorted(_)) for _ in bond_ids)

    def _get_terminal_group_matches(_mol: Mol) -> tuple[tuple, ...]:
        qsmarts = "[O,N;D1;$([O,N;D1]-[*]=[O,N;D1]),$([O,N;D1]=[*]-[O,N;D1])]~[*]"
        qsmarts = MolFromSmarts(qsmarts)
        matches = _mol.GetSubstructMatches(qsmarts)
        return _sort_bond_ids(matches)

    # sorting the atom types to use them as an index
    df["atom_types_sorted"] = df["atom_types"].apply(lambda a: tuple(sorted(a.split("--"))))
    # conjugated terminal atoms matches
    matches = _get_terminal_group_matches(mol)
    matched = df[df["atom_pair"].isin(matches)].copy()
    # min and max of lower and upper bounds
    grouped = matched.groupby("atom_types_sorted").agg({"lower_bound": np.amin, "upper_bound": np.amax})
    # updating the matches dataframe and the original dataframe
    index_orig = matched.index
    matched = matched.set_index("atom_types_sorted")
    matched.update(grouped)
    matched = matched.set_index(index_orig)
    df.update(matched)
    return df.drop(columns=["atom_types_sorted"])


def check_geometry(  # noqa: PLR0913, PLR0915
    mol_pred: Mol,
    threshold_bad_bond_length: float = 0.2,
    threshold_clash: float = 0.2,
    threshold_bad_angle: float = 0.2,
    bound_matrix_params: dict[str, Any] = bound_matrix_params,
    ignore_hydrogens: bool = True,
    sanitize: bool = True,
    symmetrize_conjugated_terminal_groups: bool = True,
) -> dict[str, Any]:
    """Use RDKit distance geometry bounds to check the geometry of a molecule.

    Args:
        mol_pred: Predicted molecule (docked ligand). Only the first conformer will be checked.
        threshold_bad_bond_length: Bond length threshold in relative percentage. 0.2 means that bonds may be up to 20%
            longer than DG bounds. Defaults to 0.2.
        threshold_clash: Threshold for how much overlap constitutes a clash. 0.2 means that the two atoms may be up to
            80% of the lower bound apart. Defaults to 0.2.
        threshold_bad_angle: Bond angle threshold in relative percentage. 0.2 means that bonds may be up to 20%
            longer than DG bounds. Defaults to 0.2.
        bound_matrix_params: Parameters passe to RDKit's GetMoleculeBoundsMatrix function.
        ignore_hydrogens: Whether to ignore hydrogens. Defaults to True.
        sanitize: Sanitize molecule before running DG module (recommended). Defaults to True.
        symmetrize_conjugated_terminal_groups: Will symmetrize the lower and upper bounds of the terminal
            conjugated bonds. Defaults to True.

    Returns:
        PoseBusters results dictionary.
    """
    mol = deepcopy(mol_pred)
    results = _empty_results.copy()

    if mol.GetNumConformers() == 0:
        logger.warning("Molecule does not have a conformer.")
        return {"results": results}

    if mol.GetNumAtoms() == 1:
        logger.warning(f"Molecule has only {mol.GetNumAtoms()} atoms.")
        results[col_angles_result] = True
        results[col_bonds_result] = True
        results[col_clash_result] = True
        return {"results": results}

    # sanitize to ensure DG works or manually process molecule
    try:
        if sanitize:
            flags = SanitizeMol(mol)
            assert flags == 0, f"Sanitization failed with flags {flags}"
    except Exception:
        return {"results": results}

    # get bonds and angles
    bond_set = sorted(_get_bond_atom_indices(mol))  # tuples
    angles = sorted(_get_angle_atom_indices(bond_set))  # triples
    angle_set = {(a[0], a[2]): a for a in angles}  # {tuples : triples}

    if len(bond_set) == 0:
        logger.warning("Molecule has no bonds.")

    # distance geometry bounds, lower triangle min distances, upper triangle max distances
    try:
        bounds = GetMoleculeBoundsMatrix(mol, **bound_matrix_params)
    except RuntimeError as e:
        logger.warning(f"RDKit distance geometry failed with error: {e}")
        return {"results": results}

    # indices
    lower_triangle_idcs = np.tril_indices(mol.GetNumAtoms(), k=-1)
    upper_triangle_idcs = (lower_triangle_idcs[1], lower_triangle_idcs[0])

    # 1,2- distances
    df_12 = pd.DataFrame()
    df_12["atom_pair"] = list(zip(*upper_triangle_idcs))  # indices have i1 < i2
    df_12["atom_types"] = [
        "--".join(tuple(mol.GetAtomWithIdx(int(j)).GetSymbol() for j in i)) for i in df_12["atom_pair"]
    ]
    df_12["angle"] = df_12["atom_pair"].apply(lambda x: angle_set.get(x, None))
    df_12["has_hydrogen"] = [_has_hydrogen(mol, i) for i in df_12["atom_pair"]]
    df_12["is_bond"] = [i in bond_set for i in df_12["atom_pair"]]
    df_12["is_angle"] = df_12["angle"].apply(lambda x: x is not None)
    df_12[col_lb] = bounds[lower_triangle_idcs]
    df_12[col_ub] = bounds[upper_triangle_idcs]

    if symmetrize_conjugated_terminal_groups:
        df_12 = symmetrize_conjugated_terminal_bonds(df_12, mol)

    # add observed dimensions
    conformer = mol.GetConformer()
    conf_distances = _pairwise_distance(conformer.GetPositions())
    df_12["conf_id"] = conformer.GetId()
    df_12["distance"] = conf_distances[lower_triangle_idcs]

    # set types
    df_12 = df_12.astype(
        {
            "has_hydrogen": "bool",
            "is_bond": "bool",
            "is_angle": "bool",
            col_lb: "float",
            col_ub: "float",
            "conf_id": "int",
            "distance": "float",
        }
    )

    # remove hydrogens if requested
    if ignore_hydrogens:
        df_12 = df_12.loc[df_12["has_hydrogen"].eq(False), :]

    # calculate violations
    df_bonds = _bond_check(df_12)
    df_clash = _clash_check(df_12)
    df_angles = _angle_check(df_12)

    # bond statistics
    results[col_n_bonds] = len(df_bonds)
    results[col_n_short_bonds] = sum(df_bonds[col_pe] < -threshold_bad_bond_length)
    results[col_n_long_bonds] = sum(df_bonds[col_pe] > threshold_bad_bond_length)
    results[col_n_good_bonds] = results[col_n_bonds] - results[col_n_short_bonds] - results[col_n_long_bonds]
    results[col_bonds_result] = results[col_n_good_bonds] == results[col_n_bonds]
    results[col_shortest_bond] = (df_bonds["distance"] / df_bonds["lower_bound"]).min()
    results[col_longest_bond] = (df_bonds["distance"] / df_bonds["upper_bound"]).max()

    # angle statistics
    results[col_n_angles] = len(df_angles)
    results[col_n_bad_angles] = sum(df_angles[col_bape] > threshold_bad_angle)
    results[col_n_good_angles] = results[col_n_angles] - results[col_n_bad_angles]
    results[col_angles_result] = results[col_n_good_angles] == results[col_n_angles]
    lb_extreme_angle = 2 - (df_angles["distance"] / df_angles["lower_bound"]).min()
    ub_extreme_angle = (df_angles["distance"] / df_angles["upper_bound"]).max()
    results[col_extremest_angle] = max(lb_extreme_angle, ub_extreme_angle)

    # steric clash statistics
    results[col_n_noncov] = len(df_clash)
    results[col_n_clashes] = sum(df_clash[col_bpe] < -threshold_clash)
    results[col_n_good_noncov] = results[col_n_noncov] - results[col_n_clashes]
    results[col_clash_result] = results[col_n_good_noncov] == results[col_n_noncov]
    results[col_closest_noncov] = (df_clash["distance"] / df_clash["lower_bound"]).min()

    return {"results": results, "details": {"bonds": df_bonds, "clash": df_clash, "angles": df_angles}}


def _bond_check(df: pd.DataFrame) -> pd.DataFrame:
    # bonds can be too short or too long
    df = df[df["is_bond"]].copy()
    df[col_pe] = df.apply(lambda x: bpe(*x[["distance", col_lb, col_ub]]), axis=1)
    return df


def _angle_check(df: pd.DataFrame) -> pd.DataFrame:
    # angles have no direction (we do not know if larger or bigger beyond bounds)
    df = df[(~df["is_bond"]) & (df["is_angle"])].copy()
    df[col_bape] = df.apply(lambda x: bape(*x[["distance", col_lb, col_ub]]), axis=1)
    return df


def _clash_check(df: pd.DataFrame) -> pd.DataFrame:
    # clash is only when lower bound is violated
    df = df[(~df["is_bond"]) & (~df["is_angle"])].copy()

    def _lb_pe(value: float, lower_bound: float) -> float:
        if value >= lower_bound:
            return 0.0
        return pe(value, lower_bound)

    df[col_bpe] = df.apply(lambda x: _lb_pe(*x[["distance", col_lb]]), axis=1)
    return df


def _get_bond_atom_indices(mol: Mol) -> list[tuple[int, int]]:
    bonds = []
    for bond in mol.GetBonds():
        bond_tuple = (bond.GetBeginAtomIdx(), bond.GetEndAtomIdx())
        bond_tuple = _sort_bond(bond_tuple)
        bonds.append(bond_tuple)
    return bonds


def _get_angle_atom_indices(bonds: list[tuple[int, int]]) -> list[tuple[int, int, int]]:
    """Check all combinations of bonds to generate list of molecule angles."""
    angles = []
    bonds = list(bonds)
    for i in range(len(bonds)):
        for j in range(i + 1, len(bonds)):
            angle = _two_bonds_to_angle(bonds[i], bonds[j])
            if angle is not None:
                angles.append(angle)
    return angles


def _two_bonds_to_angle(bond1: tuple[int, int], bond2: tuple[int, int]) -> None | tuple[int, int, int]:
    set1 = set(bond1)
    set2 = set(bond2)
    all_atoms = set1 | set2
    # angle requires two bonds to share exactly one atom, that is we must have 3 atoms
    if len(all_atoms) != 3:  # noqa: PLR2004
        return None
    # find shared atom
    shared_atom = set1 & set2
    other_atoms = all_atoms - shared_atom
    return (min(other_atoms), shared_atom.pop(), max(other_atoms))


def _sort_bond(bond: tuple[int, int]) -> tuple[int, int]:
    return (min(bond), max(bond))


def _pairwise_distance(x: np.ndarray) -> np.ndarray:
    return np.linalg.norm(x[:, None, :] - x[None, :, :], axis=-1)


def _has_hydrogen(mol: Mol, idcs: Iterable[int]) -> bool:
    return any(_is_hydrogen(mol, idx) for idx in idcs)


def _is_hydrogen(mol: Mol, idx: int) -> bool:
    return mol.GetAtomWithIdx(int(idx)).GetAtomicNum() == 1


In [ ]:
from copy import deepcopy
import math

from rdkit import Chem
from rdkit.Chem import AllChem, Lipinski
from rdkit.Chem.rdDistGeom import ETKDGv3
from rdkit.Chem.rdForceFieldHelpers import (
    UFFHasAllMoleculeParams,
    UFFOptimizeMoleculeConfs,
)


def _safe_set(params, name, value):
    """Safely set RDKit EmbedParameters attribute."""
    if hasattr(params, name):
        try:
            setattr(params, name, value)
        except Exception:
            pass


def _get_ring_profile(mol):
    ri = mol.GetRingInfo()
    rings = list(ri.AtomRings())
    ring_sizes = [len(r) for r in rings]

    n_aromatic_rings = 0
    for ring in rings:
        if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
            n_aromatic_rings += 1

    return {
        "n_heavy": mol.GetNumHeavyAtoms(),
        "n_atoms": mol.GetNumAtoms(),
        "n_rotatable_bonds": Lipinski.NumRotatableBonds(mol),
        "n_rings": len(rings),
        "ring_sizes": ring_sizes,
        "max_ring_size": max(ring_sizes) if ring_sizes else 0,
        "n_macrocycles": sum(1 for s in ring_sizes if s >= 8),
        "n_small_rings": sum(1 for s in ring_sizes if s <= 4),
        "n_aromatic_rings": n_aromatic_rings,
    }


def _is_large_or_complex(profile):
    """
    Decide whether to use the large/complex molecule embedding strategy.
    """
    reasons = []

    if profile["max_ring_size"] >= 8:
        reasons.append("macrocycle")

    if profile["n_heavy"] >= 60:
        reasons.append("large_molecule")

    if profile["n_rings"] >= 10:
        reasons.append("many_rings")

    if profile["n_rotatable_bonds"] >= 20:
        reasons.append("many_rotatable_bonds")

    if profile["n_heavy"] >= 45 and profile["n_rings"] >= 6:
        reasons.append("large_fused_or_polycyclic_system")

    return len(reasons) > 0, reasons


def _make_normal_etkdg(num_threads=0, random_seed=42):
    """
    Original/simple ETKDGv3 strategy for ordinary molecules.
    """
    p = ETKDGv3()
    p.randomSeed = random_seed
    p.verbose = False
    p.useRandomCoords = True
    p.numThreads = num_threads
    return p


def _make_large_complex_etkdg(
    num_threads=0,
    random_seed=42,
    max_iters_embed=2000,
    timeout=30,
):
    """
    Macrocycle-aware ETKDGv3 strategy for large, macrocyclic, or complex molecules.
    Only one strategy, no multiple fallback recipes.
    """
    p = ETKDGv3()

    _safe_set(p, "randomSeed", random_seed)
    _safe_set(p, "verbose", False)
    _safe_set(p, "numThreads", num_threads)

    # More robust initialization for large rings / complex topology
    _safe_set(p, "useRandomCoords", True)
    _safe_set(p, "boxSizeMult", 6.0)

    # ETKDG chemical knowledge
    _safe_set(p, "useExpTorsionAnglePrefs", True)
    _safe_set(p, "useBasicKnowledge", True)
    _safe_set(p, "enforceChirality", True)

    # Ring / macrocycle-specific settings
    _safe_set(p, "useMacrocycleTorsions", True)
    _safe_set(p, "useMacrocycle14config", True)
    _safe_set(p, "useSmallRingTorsions", True)

    # Avoid excessive running time
    _safe_set(p, "maxIterations", max_iters_embed)
    _safe_set(p, "timeout", timeout)

    # Remove near-duplicate conformers
    _safe_set(p, "pruneRmsThresh", 0.35)
    _safe_set(p, "onlyHeavyAtomsForRMS", True)

    return p


def _choose_n_confs_two_mode(n_confs, profile, is_complex):
    """
    Ordinary molecules use requested n_confs.
    Large/complex molecules use fewer conformers for speed.
    """
    if not is_complex:
        return n_confs

    n_heavy = profile["n_heavy"]
    n_rings = profile["n_rings"]
    max_ring = profile["max_ring_size"]
    n_rot = profile["n_rotatable_bonds"]

    if n_heavy >= 100 or n_rings >= 25:
        return min(n_confs, 3)

    if n_heavy >= 80 or n_rings >= 18 or max_ring >= 16:
        return min(n_confs, 5)

    if n_heavy >= 60 or n_rot >= 25 or max_ring >= 12:
        return min(n_confs, 10)

    return min(n_confs, 20)


def _keep_only_best_conformer(mol_with_confs, best_conf_id):
    """
    Keep only one conformer without deepcopying RDKit Conformer object.
    """
    mol_best = Chem.Mol(mol_with_confs)

    best_conf_id = int(best_conf_id)

    for conf in list(mol_best.GetConformers()):
        if conf.GetId() != best_conf_id:
            mol_best.RemoveConformer(conf.GetId())

    return mol_best


def cal_e(
    smiles,
    n_confs: int = 50,
    num_threads: int = 4,
    max_iters_embed: int = 2000,
    max_iters_uff: int = 200,
    timeout: int = 30,
    random_seed: int = 42,
):
    """
    Two-mode UFF conformer generation.

    Mode 1:
        ordinary molecule -> original ETKDGv3 + UFF

    Mode 2:
        macrocycle / large / complex molecule -> macrocycle-aware ETKDGv3 + UFF
    """

    # 1. RDKit parse and sanitize
    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return {
                "success": False,
                "mol": None,
                "error": "RDKit failed to parse SMILES",
            }

        Chem.SanitizeMol(mol)

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"RDKit parse/sanitization failed: {e}",
        }

    # 2. Decide normal or complex mode
    profile = _get_ring_profile(mol)
    is_complex, complexity_reasons = _is_large_or_complex(profile)

    if is_complex:
        embedding_mode = "large_complex_macrocycle_ETKDGv3"
        etkdg = _make_large_complex_etkdg(
            num_threads=num_threads,
            random_seed=random_seed,
            max_iters_embed=max_iters_embed,
            timeout=timeout,
        )
    else:
        embedding_mode = "normal_ETKDGv3"
        etkdg = _make_normal_etkdg(
            num_threads=num_threads,
            random_seed=random_seed,
        )

    n_confs_eff = _choose_n_confs_two_mode(
        n_confs=n_confs,
        profile=profile,
        is_complex=is_complex,
    )

    # 3. Add hydrogens and check UFF support
    mol_h = Chem.Mol(mol)
    mol_h = Chem.AddHs(mol_h)
    mol_h.RemoveAllConformers()

    if not UFFHasAllMoleculeParams(mol_h):
        return {
            "success": False,
            "mol": None,
            "error": "UFF parameters missing for molecule",
            "embedding_mode": embedding_mode,
            "complexity_reasons": complexity_reasons,
            "n_confs_requested": n_confs,
            "n_confs_used": n_confs_eff,
            **profile,
        }

    # 4. Embed conformers
    try:
        cids = list(
            AllChem.EmbedMultipleConfs(
                mol_h,
                numConfs=n_confs_eff,
                params=etkdg,
            )
        )

        if len(cids) == 0:
            return {
                "success": False,
                "mol": None,
                "error": "Failed to generate conformations",
                "embedding_mode": embedding_mode,
                "complexity_reasons": complexity_reasons,
                "n_confs_requested": n_confs,
                "n_confs_used": n_confs_eff,
                **profile,
            }

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"ETKDG embedding failed: {e}",
            "embedding_mode": embedding_mode,
            "complexity_reasons": complexity_reasons,
            "n_confs_requested": n_confs,
            "n_confs_used": n_confs_eff,
            **profile,
        }

    # 5. UFF optimization
    try:
        opt_results = UFFOptimizeMoleculeConfs(
            mol_h,
            numThreads=num_threads,
            maxIters=max_iters_uff,
        )

        records = []

        for conf_id, (not_converged, energy) in zip(cids, opt_results):
            if energy is None or not math.isfinite(float(energy)):
                continue

            records.append({
                "conf_id": int(conf_id),
                "not_converged": int(not_converged),
                "energy": float(energy),
            })

        if not records:
            return {
                "success": False,
                "mol": None,
                "error": "No valid UFF energy was obtained",
                "embedding_mode": embedding_mode,
                "complexity_reasons": complexity_reasons,
                "n_confs_requested": n_confs,
                "n_confs_used": n_confs_eff,
                "n_confs_generated": len(cids),
                **profile,
            }

        # Prefer converged conformers
        converged = [r for r in records if r["not_converged"] == 0]
        pool = converged if converged else records

        best = min(pool, key=lambda x: x["energy"])

        mol_best = _keep_only_best_conformer(
            mol_with_confs=mol_h,
            best_conf_id=best["conf_id"],
        )

        n_heavy = mol.GetNumHeavyAtoms()

        return {
            "success": True,
            "mol": mol_best,
            "energy": best["energy"],
            "energy_per_heavy_atom": best["energy"] / max(n_heavy, 1),
            "conf_id": best["conf_id"],
            "not_converged": best["not_converged"],

            "embedding_mode": embedding_mode,
            "forcefield": "UFF",
            "is_large_or_complex": is_complex,
            "complexity_reasons": complexity_reasons,

            "n_confs_requested": n_confs,
            "n_confs_used": n_confs_eff,
            "n_confs_generated": len(cids),
            "n_confs_converged": len(converged),
            "all_energies": [r["energy"] for r in records],

            "error": None,
            **profile,
        }

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"UFF optimization failed: {e}",
            "embedding_mode": embedding_mode,
            "complexity_reasons": complexity_reasons,
            "n_confs_requested": n_confs,
            "n_confs_used": n_confs_eff,
            "n_confs_generated": len(cids),
            **profile,
        }

# =========================
# 2. Geometry checks
# =========================

def pair_distance(conf, i, j):
    pi = conf.GetAtomPosition(int(i))
    pj = conf.GetAtomPosition(int(j))
    return np.linalg.norm(np.array([pi.x - pj.x, pi.y - pj.y, pi.z - pj.z]))


def get_bond_pairs(mol):
    pairs = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        pairs.append(tuple(sorted((i, j))))
    return set(pairs)


def get_angle_pairs_from_bonds(bond_pairs):
    angle_pairs = {}
    bonds = list(bond_pairs)

    for a in range(len(bonds)):
        for b in range(a + 1, len(bonds)):
            s1, s2 = set(bonds[a]), set(bonds[b])
            shared = s1 & s2
            if len(shared) != 1:
                continue

            center = list(shared)[0]
            ends = list((s1 | s2) - shared)
            pair = tuple(sorted(ends))
            angle_pairs[pair] = (ends[0], center, ends[1])

    return angle_pairs


def has_hydrogen(mol, pair):
    return any(mol.GetAtomWithIdx(int(i)).GetAtomicNum() == 1 for i in pair)


def check_geometry(
    mol3d,
    threshold_bond=0.25,
    threshold_angle=0.25,
    threshold_clash=0.30,
    ignore_hydrogens=True,
):
    try:
        Chem.SanitizeMol(mol3d)
        conf = mol3d.GetConformer()
        bounds = GetMoleculeBoundsMatrix(
            mol3d,
            set15bounds=True,
            scaleVDW=True,
            doTriangleSmoothing=True,
            useMacrocycle14config=False,
        )
    except Exception as e:
        return {"success": False, "error": f"geometry setup failed: {e}"}

    n_atoms = mol3d.GetNumAtoms()
    bond_pairs = get_bond_pairs(mol3d)
    angle_pairs = get_angle_pairs_from_bonds(bond_pairs)

    n_bonds = n_short = n_long = 0
    n_angles = n_bad_angles = 0
    n_noncov = n_clashes = 0

    shortest_bond_ratio = np.nan
    longest_bond_ratio = np.nan
    closest_noncov_ratio = np.nan

    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            pair = (i, j)

            if ignore_hydrogens and has_hydrogen(mol3d, pair):
                continue

            d = pair_distance(conf, i, j)
            lb = bounds[j, i]
            ub = bounds[i, j]

            if pair in bond_pairs:
                n_bonds += 1
                shortest_bond_ratio = np.nanmin([shortest_bond_ratio, d / lb])
                longest_bond_ratio = np.nanmax([longest_bond_ratio, d / ub])

                if d < (1 - threshold_bond) * lb:
                    n_short += 1
                elif d > (1 + threshold_bond) * ub:
                    n_long += 1

            elif pair in angle_pairs:
                n_angles += 1

                if d < (1 - threshold_angle) * lb or d > (1 + threshold_angle) * ub:
                    n_bad_angles += 1

            else:
                n_noncov += 1
                closest_noncov_ratio = np.nanmin([closest_noncov_ratio, d / lb])

                if d < (1 - threshold_clash) * lb:
                    n_clashes += 1

    return {
        "success": True,
        "number_bonds": n_bonds,
        "number_short_bonds": n_short,
        "number_long_bonds": n_long,
        "bond_lengths_pass": (n_short == 0 and n_long == 0),
        "shortest_bond_relative_length": shortest_bond_ratio,
        "longest_bond_relative_length": longest_bond_ratio,

        "number_angles": n_angles,
        "number_bad_angles": n_bad_angles,
        "bond_angles_pass": (n_bad_angles == 0),

        "number_noncov_pairs": n_noncov,
        "number_clashes": n_clashes,
        "internal_clash_pass": (n_clashes == 0),
        "shortest_noncovalent_relative_distance": closest_noncov_ratio,
        "error": None,
    }


# =========================
# 3. Flatness checks
# =========================

def coords_of_atoms(mol, atom_indices):
    conf = mol.GetConformer()
    return np.array([
        [
            conf.GetAtomPosition(int(i)).x,
            conf.GetAtomPosition(int(i)).y,
            conf.GetAtomPosition(int(i)).z,
        ]
        for i in atom_indices
    ])


def max_distance_to_plane(coords):
    coords = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(coords)
    normal = vh[-1]
    distances = np.abs(np.dot(coords, normal))
    return float(distances.max())


def get_flat_groups(mol):
    groups = []

    # 5/6-membered aromatic rings
    ring_info = mol.GetRingInfo()
    for ring in ring_info.AtomRings():
        if len(ring) in (5, 6):
            if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
                groups.append(("aromatic_ring", tuple(ring)))

    # non-aromatic aliphatic C=C and its neighbours
    for bond in mol.GetBonds():
        if bond.GetBondType() != Chem.rdchem.BondType.DOUBLE:
            continue
        if bond.GetIsAromatic() or bond.IsInRing():
            continue

        a1 = bond.GetBeginAtom()
        a2 = bond.GetEndAtom()

        if a1.GetSymbol() != "C" or a2.GetSymbol() != "C":
            continue

        group = {a1.GetIdx(), a2.GetIdx()}
        group.update([n.GetIdx() for n in a1.GetNeighbors()])
        group.update([n.GetIdx() for n in a2.GetNeighbors()])

        if len(group) >= 4:
            groups.append(("double_bond", tuple(sorted(group))))

    return groups


def check_flatness(mol3d, threshold_flatness=0.25):
    try:
        Chem.SanitizeMol(mol3d)
        groups = get_flat_groups(mol3d)
    except Exception as e:
        return {"success": False, "error": f"flatness setup failed: {e}"}

    max_distances = []
    records = []

    for group_type, atom_indices in groups:
        coords = coords_of_atoms(mol3d, atom_indices)
        max_d = max_distance_to_plane(coords)
        passed = max_d <= threshold_flatness

        records.append({
            "type": group_type,
            "atom_indices": atom_indices,
            "max_distance": max_d,
            "pass": passed,
        })
        max_distances.append(max_d)

    return {
        "success": True,
        "num_systems_checked": len(groups),
        "num_systems_passed": sum(r["pass"] for r in records),
        "max_flatness_distance": max(max_distances) if max_distances else np.nan,
        "flatness_pass": all(r["pass"] for r in records) if records else True,
        "details": records,
        "error": None,
    }


# =========================
# 4. Full assessment for one SMILES
# =========================

def assess_one_smiles_3d(smiles, n_confs=50, num_threads=0):
    conf_res = cal_e(
        smiles,
        n_confs=n_confs,
        num_threads=num_threads,
    )

    if not conf_res["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": conf_res["error"],
        }

    mol3d = conf_res["mol"]

    geo = check_geometry(mol3d)
    flat = check_flatness(mol3d)

    if not geo["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": geo["error"],
        }

    if not flat["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": flat["error"],
        }

    three_d_pass = (
        geo["bond_lengths_pass"]
        and geo["bond_angles_pass"]
        and geo["internal_clash_pass"]
        and flat["flatness_pass"]
    )

    return {
        "smiles": smiles,
        "success": True,
        "three_d_pass": three_d_pass,

        "energy": conf_res["energy"],
        
        "n_confs_generated": conf_res["n_confs_generated"],
        "n_confs_converged": conf_res["n_confs_converged"],

        "bond_lengths_pass": geo["bond_lengths_pass"],
        "bond_angles_pass": geo["bond_angles_pass"],
        "internal_clash_pass": geo["internal_clash_pass"],
        "flatness_pass": flat["flatness_pass"],

        "number_short_bonds": geo["number_short_bonds"],
        "number_long_bonds": geo["number_long_bonds"],
        "number_bad_angles": geo["number_bad_angles"],
        "number_clashes": geo["number_clashes"],
        "num_flat_systems_checked": flat["num_systems_checked"],
        "num_flat_systems_passed": flat["num_systems_passed"],
        "max_flatness_distance": flat["max_flatness_distance"],

        "mol3d": mol3d,
        "error": None,
    }


# =========================
# 5. Batch assessment
# =========================

import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm
from rdkit import RDLogger


def _assess_one_smiles_3d_worker(args):
    """
    Worker function for multiprocessing.
    注意：不要返回 RDKit Mol 对象，否则可能 pickle 失败或很慢。
    """
    idx, smi, n_confs, num_threads_per_mol = args

    try:
        RDLogger.DisableLog("rdApp.warning")

        res = assess_one_smiles_3d(
            smi,
            n_confs=n_confs,
            num_threads=num_threads_per_mol,
        )

        is_success = bool(res.get("success", False))
        three_d_pass = bool(res.get("three_d_pass", False))
        is_bad = (not is_success) or (not three_d_pass)

        if not is_success:
            failure_stage = "3D generation or geometry setup failed"
        elif not three_d_pass:
            failure_stage = "3D geometry check failed"
        else:
            failure_stage = None

        # 关键：去掉 RDKit Mol 对象
        row = {
            k: v for k, v in res.items()
            if k not in ("mol3d", "mol")
        }

        row["index"] = idx
        row["smiles"] = smi
        row["is_unreasonable_3d"] = is_bad
        row["failure_stage"] = failure_stage

        return row

    except Exception as e:
        return {
            "index": idx,
            "smiles": smi,
            "success": False,
            "three_d_pass": False,
            "is_unreasonable_3d": True,
            "failure_stage": "worker exception",
            "error": str(e),
        }


def assess_smiles_list_3d_parallel(
    smiles_list,
    n_confs=50,
    n_workers=32,
    num_threads_per_mol=1,
    save_all_csv_path=None,
    save_bad_csv_path=None,
    print_bad=True,
):
    """
    Parallel version of assess_smiles_list_3d.

    Parameters
    ----------
    smiles_list:
        list of SMILES
    n_confs:
        conformers per molecule
    n_workers:
        number of parallel processes
    num_threads_per_mol:
        RDKit threads inside each molecule. Usually set to 1 or 2.
    """

    tasks = [
        (idx, smi, n_confs, num_threads_per_mol)
        for idx, smi in enumerate(smiles_list)
    ]

    rows = [None] * len(smiles_list)
    bad_smiles = []
    bad_rows = []
    bad_count = 0

    with ProcessPoolExecutor(max_workers=n_workers) as executor:
        futures = [
            executor.submit(_assess_one_smiles_3d_worker, task)
            for task in tasks
        ]

        for fut in tqdm(as_completed(futures), total=len(futures)):
            row = fut.result()
            idx = row["index"]
            rows[idx] = row

            if row.get("is_unreasonable_3d", False):
                bad_count += 1
                bad_smiles.append(row["smiles"])
                bad_rows.append(row)

                if print_bad:
                    print(
                        f"[UNREASONABLE_3D] index={idx} | "
                        f"stage={row.get('failure_stage')} | "
                        f"error={row.get('error')} | "
                        f"SMILES:{row.get('smiles')} | "
                        f"count={bad_count}",
                        flush=True,
                    )

    rows = [r for r in rows if r is not None]

    df_all = pd.DataFrame(rows)
    df_bad = pd.DataFrame(bad_rows)

    if save_all_csv_path is not None:
        df_all.to_csv(save_all_csv_path, index=False)

    if save_bad_csv_path is not None:
        df_bad.to_csv(save_bad_csv_path, index=False)

    return {
        "df_all": df_all,
        "df_bad": df_bad,
        "bad_smiles": bad_smiles,
    }

In [ ]:

from __future__ import annotations

import logging
from copy import deepcopy
import numpy as np
from functools import lru_cache

cache = lru_cache(maxsize=None)
from math import isfinite

from rdkit import ForceField  # noqa: F401
from rdkit.Chem.inchi import InchiReadWriteError, MolFromInchi
from rdkit.Chem.rdchem import Mol
from rdkit.Chem.rdDistGeom import EmbedMultipleConfs, ETKDGv3
from rdkit.Chem.rdForceFieldHelpers import (
    UFFGetMoleculeForceField,
    UFFHasAllMoleculeParams,
    UFFOptimizeMoleculeConfs,
)
from rdkit.Chem.rdmolops import AddHs, AssignStereochemistryFrom3D, SanitizeMol

import pandas as pd
import logging
import re
from copy import deepcopy
from re import findall

from rdkit.Chem.inchi import MolFromInchi, MolToInchi
from rdkit.Chem.MolStandardize.rdMolStandardize import Uncharger
from rdkit.Chem.rdchem import AtomValenceException, Mol
from rdkit.Chem.rdmolops import AssignStereochemistryFrom3D, RemoveHs, RemoveStereochemistry, SanitizeMol

logger = logging.getLogger(__name__)



def get_conf_energy(mol: Mol, conf_id: int = -1) -> float:
    """Get energy of a conformation."""
    assert mol is not None

    mol = AddHs(mol, addCoords=True)
    with CaptureLogger():
        uff = UFFGetMoleculeForceField(mol, confId=conf_id)
        e_uff = uff.CalcEnergy()
    return e_uff


def new_conformation(
    mol: Mol, n_confs: int = 1, num_threads: int = 0, energy_minimization=True
) -> dict[str, Mol | list[float]]:
    """Generate new conformation(s) for a molecule."""
    assert mol is not None

    etkdg = ETKDGv3()
    etkdg.randomSeed = 42  # type: ignore[assignment]
    etkdg.verbose = False  # type: ignore[assignment]
    etkdg.useRandomCoords = True  # type: ignore[assignment]
    etkdg.numThreads = num_threads  # type: ignore[assignment]

    # prep mol
    mol_etkdg = deepcopy(mol)
    mol_etkdg = AddHs(mol_etkdg, addCoords=True)
    AssignStereochemistryFrom3D(mol_etkdg, replaceExistingTags=True)
    mol_etkdg.RemoveAllConformers()

    # etkdg
    with CaptureLogger():
        cids_etkdg = EmbedMultipleConfs(mol_etkdg, n_confs, etkdg)
        assert len(cids_etkdg) == n_confs, "Failed to generate conformations."

    # energy minimization
    if energy_minimization:
        with CaptureLogger():
            energy_uff = UFFOptimizeMoleculeConfs(mol_etkdg, numThreads=num_threads)
        energy_uff = [v[1] for v in energy_uff]
    else:
        energy_uff = [get_conf_energy(mol_etkdg, conf_id) for conf_id in cids_etkdg]

    return {"mol": mol_etkdg, "energies": energy_uff}

@cache
def get_energies(inchi: str, n_confs: int = 50, num_threads: int = 0) -> list[float]:
    """Get energies of an ensemble of molecule conformations."""
    with CaptureLogger():
        mol = MolFromInchi(inchi)
    return new_conformation(mol, n_confs, num_threads)["energies"]


def get_inchi(mol: Mol, inchi_strict: bool = False) -> str:
    """Get inchi of a molecule."""

    with CaptureLogger() as log:
        SanitizeMol(mol)
        inchi = str(MolToInchi(mol, treatWarningAsError=inchi_strict))
        # check inchi because inchi generation does not raise an error if the inchi is invalid
        if MolFromInchi(inchi, sanitize=True) is None:
            warnings = re.split(r"\[.+?\] WARNING: ", log.get("WARNING", ""))[1:]
            errors = re.split(r"\[.+?\] ERROR: ", log.get("ERROR", ""))[1:]
            message = f"InChI ERRORs: {'; '.join(errors)} WARNINGs: {'; '.join(warnings)}"
            raise Exception(message)
    return inchi
# https://github.com/rdkit/rdkit/discussions/5435
class CaptureLogger(logging.Handler):
    """Helper class that captures Python logger output."""

    def __init__(self, level: str | None = None) -> None:
        """Initialize logger."""
        super().__init__(level=logging.NOTSET if level is None else level)
        self.logs: dict[str, str] = {}
        self.devnull = open(os.devnull, "w")
        rdkit.log_handler.setStream(self.devnull)
        rdkit.logger.addHandler(self)

    def __enter__(self) -> dict[str, str]:
        """Enter context manager."""
        return self.logs

    def __exit__(self, exc_type: type[BaseException] | None, exc_val: BaseException | None, exc_tb: Any) -> None:
        """Exit context manager."""
        self.release()

    def handle(self, record: logging.LogRecord) -> bool:
        """Handle log record."""
        key = record.levelname
        val = self.format(record)
        self.logs[key] = self.logs.get(key, "") + val
        return False

    def release(self) -> None:
        """Release logger."""
        rdkit.log_handler.setStream(sys.stderr)
        rdkit.logger.removeHandler(self)
        self.devnull.close()

    def emit(self, record: logging.LogRecord) -> None:
        """Emit a log record."""
def add_hydrogens_with_uff_positions(mol: Mol) -> tuple[Mol, float, float]:
    SanitizeMol(mol)
    indices_before = {atom.GetIdx() for atom in mol.GetAtoms()}
    mol = hydrate_radicals(mol)
    mol = AddHs(mol, addCoords=True)
    return optimize_positions(mol, indices_before)



In [ ]:
import math
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
from rdkit.Chem.rdDistGeom import ETKDGv3
from rdkit.Chem.rdForceFieldHelpers import (
    UFFHasAllMoleculeParams,
    UFFOptimizeMoleculeConfs,
)


# ============================================================
# 1. Basic ETKDGv3 + UFF
# ============================================================

def cal_e(
    smiles,
    n_confs: int = 50,
    num_threads: int = 1,
    max_iters_uff: int = 200,
    random_seed: int = 42,
):
    """
    Basic ETKDGv3 + UFF conformer generation.

    SMILES
    -> RDKit sanitize
    -> AddHs
    -> ETKDGv3 conformer generation
    -> UFF optimization
    -> keep the lowest-energy conformer
    """

    # 1. Parse and sanitize
    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            return {
                "success": False,
                "mol": None,
                "error": "RDKit failed to parse SMILES",
            }

        Chem.SanitizeMol(mol)

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"RDKit parse/sanitization failed: {e}",
        }

    # 2. Add Hs
    try:
        mol_h = Chem.Mol(mol)
        mol_h = Chem.AddHs(mol_h)
        mol_h.RemoveAllConformers()

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"AddHs failed: {e}",
        }

    # 3. Check UFF support
    try:
        if not UFFHasAllMoleculeParams(mol_h):
            return {
                "success": False,
                "mol": None,
                "error": "UFF parameters missing for molecule",
                "n_heavy": mol.GetNumHeavyAtoms(),
            }

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"UFF parameter check failed: {e}",
            "n_heavy": mol.GetNumHeavyAtoms(),
        }

    # 4. Basic ETKDGv3 embedding
    try:
        etkdg = ETKDGv3()
        etkdg.randomSeed = random_seed
        etkdg.verbose = False
        etkdg.useRandomCoords = True
        etkdg.numThreads = num_threads

        cids = list(
            AllChem.EmbedMultipleConfs(
                mol_h,
                numConfs=n_confs,
                params=etkdg,
            )
        )

        if len(cids) == 0:
            return {
                "success": False,
                "mol": None,
                "error": "Failed to generate conformations",
                "n_heavy": mol.GetNumHeavyAtoms(),
                "n_confs_requested": n_confs,
                "n_confs_generated": 0,
            }

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"ETKDG embedding failed: {e}",
            "n_heavy": mol.GetNumHeavyAtoms(),
            "n_confs_requested": n_confs,
        }

    # 5. UFF optimization
    try:
        opt_results = UFFOptimizeMoleculeConfs(
            mol_h,
            numThreads=num_threads,
            maxIters=max_iters_uff,
        )

        records = []

        for conf_id, (not_converged, energy) in zip(cids, opt_results):
            if energy is None:
                continue

            energy = float(energy)

            if not math.isfinite(energy):
                continue

            records.append({
                "conf_id": int(conf_id),
                "not_converged": int(not_converged),
                "energy": energy,
            })

        if not records:
            return {
                "success": False,
                "mol": None,
                "error": "No valid UFF energy was obtained",
                "n_heavy": mol.GetNumHeavyAtoms(),
                "n_confs_requested": n_confs,
                "n_confs_generated": len(cids),
            }

        # Prefer converged conformers
        converged = [r for r in records if r["not_converged"] == 0]
        pool = converged if converged else records

        best = min(pool, key=lambda x: x["energy"])

        # Keep only best conformer
        mol_best = Chem.Mol(mol_h)

        best_conf_id = int(best["conf_id"])

        for conf in list(mol_best.GetConformers()):
            if conf.GetId() != best_conf_id:
                mol_best.RemoveConformer(conf.GetId())

        n_heavy = mol.GetNumHeavyAtoms()

        return {
            "success": True,
            "mol": mol_best,
            "energy": best["energy"],
            "energy_per_heavy_atom": best["energy"] / max(n_heavy, 1),
            "conf_id": best["conf_id"],
            "not_converged": best["not_converged"],
            "forcefield": "UFF",
            "embedding_method": "basic_ETKDGv3",
            "n_heavy": n_heavy,
            "n_confs_requested": n_confs,
            "n_confs_generated": len(cids),
            "n_confs_converged": len(converged),
            "all_energies": [r["energy"] for r in records],
            "error": None,
        }

    except Exception as e:
        return {
            "success": False,
            "mol": None,
            "error": f"UFF optimization failed: {e}",
            "n_heavy": mol.GetNumHeavyAtoms(),
            "n_confs_requested": n_confs,
            "n_confs_generated": len(cids),
        }
        
        

# =========================
# 2. Geometry checks
# =========================

def pair_distance(conf, i, j):
    pi = conf.GetAtomPosition(int(i))
    pj = conf.GetAtomPosition(int(j))
    return np.linalg.norm(np.array([pi.x - pj.x, pi.y - pj.y, pi.z - pj.z]))


def get_bond_pairs(mol):
    pairs = []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        pairs.append(tuple(sorted((i, j))))
    return set(pairs)


def get_angle_pairs_from_bonds(bond_pairs):
    angle_pairs = {}
    bonds = list(bond_pairs)

    for a in range(len(bonds)):
        for b in range(a + 1, len(bonds)):
            s1, s2 = set(bonds[a]), set(bonds[b])
            shared = s1 & s2
            if len(shared) != 1:
                continue

            center = list(shared)[0]
            ends = list((s1 | s2) - shared)
            pair = tuple(sorted(ends))
            angle_pairs[pair] = (ends[0], center, ends[1])

    return angle_pairs


def has_hydrogen(mol, pair):
    return any(mol.GetAtomWithIdx(int(i)).GetAtomicNum() == 1 for i in pair)


def check_geometry(
    mol3d,
    threshold_bond=0.25,
    threshold_angle=0.25,
    threshold_clash=0.30,
    ignore_hydrogens=True,
):
    try:
        Chem.SanitizeMol(mol3d)
        conf = mol3d.GetConformer()
        bounds = GetMoleculeBoundsMatrix(
            mol3d,
            set15bounds=True,
            scaleVDW=True,
            doTriangleSmoothing=True,
            useMacrocycle14config=False,
        )
    except Exception as e:
        return {"success": False, "error": f"geometry setup failed: {e}"}

    n_atoms = mol3d.GetNumAtoms()
    bond_pairs = get_bond_pairs(mol3d)
    angle_pairs = get_angle_pairs_from_bonds(bond_pairs)

    n_bonds = n_short = n_long = 0
    n_angles = n_bad_angles = 0
    n_noncov = n_clashes = 0

    shortest_bond_ratio = np.nan
    longest_bond_ratio = np.nan
    closest_noncov_ratio = np.nan

    for i in range(n_atoms):
        for j in range(i + 1, n_atoms):
            pair = (i, j)

            if ignore_hydrogens and has_hydrogen(mol3d, pair):
                continue

            d = pair_distance(conf, i, j)
            lb = bounds[j, i]
            ub = bounds[i, j]

            if pair in bond_pairs:
                n_bonds += 1
                shortest_bond_ratio = np.nanmin([shortest_bond_ratio, d / lb])
                longest_bond_ratio = np.nanmax([longest_bond_ratio, d / ub])

                if d < (1 - threshold_bond) * lb:
                    n_short += 1
                elif d > (1 + threshold_bond) * ub:
                    n_long += 1

            elif pair in angle_pairs:
                n_angles += 1

                if d < (1 - threshold_angle) * lb or d > (1 + threshold_angle) * ub:
                    n_bad_angles += 1

            else:
                n_noncov += 1
                closest_noncov_ratio = np.nanmin([closest_noncov_ratio, d / lb])

                if d < (1 - threshold_clash) * lb:
                    n_clashes += 1

    return {
        "success": True,
        "number_bonds": n_bonds,
        "number_short_bonds": n_short,
        "number_long_bonds": n_long,
        "bond_lengths_pass": (n_short == 0 and n_long == 0),
        "shortest_bond_relative_length": shortest_bond_ratio,
        "longest_bond_relative_length": longest_bond_ratio,

        "number_angles": n_angles,
        "number_bad_angles": n_bad_angles,
        "bond_angles_pass": (n_bad_angles == 0),

        "number_noncov_pairs": n_noncov,
        "number_clashes": n_clashes,
        "internal_clash_pass": (n_clashes == 0),
        "shortest_noncovalent_relative_distance": closest_noncov_ratio,
        "error": None,
    }


# =========================
# 3. Flatness checks
# =========================

def coords_of_atoms(mol, atom_indices):
    conf = mol.GetConformer()
    return np.array([
        [
            conf.GetAtomPosition(int(i)).x,
            conf.GetAtomPosition(int(i)).y,
            conf.GetAtomPosition(int(i)).z,
        ]
        for i in atom_indices
    ])


def max_distance_to_plane(coords):
    coords = coords - coords.mean(axis=0)
    _, _, vh = np.linalg.svd(coords)
    normal = vh[-1]
    distances = np.abs(np.dot(coords, normal))
    return float(distances.max())


def get_flat_groups(mol):
    groups = []

    # 5/6-membered aromatic rings
    ring_info = mol.GetRingInfo()
    for ring in ring_info.AtomRings():
        if len(ring) in (5, 6):
            if all(mol.GetAtomWithIdx(i).GetIsAromatic() for i in ring):
                groups.append(("aromatic_ring", tuple(ring)))

    # non-aromatic aliphatic C=C and its neighbours
    for bond in mol.GetBonds():
        if bond.GetBondType() != Chem.rdchem.BondType.DOUBLE:
            continue
        if bond.GetIsAromatic() or bond.IsInRing():
            continue

        a1 = bond.GetBeginAtom()
        a2 = bond.GetEndAtom()

        if a1.GetSymbol() != "C" or a2.GetSymbol() != "C":
            continue

        group = {a1.GetIdx(), a2.GetIdx()}
        group.update([n.GetIdx() for n in a1.GetNeighbors()])
        group.update([n.GetIdx() for n in a2.GetNeighbors()])

        if len(group) >= 4:
            groups.append(("double_bond", tuple(sorted(group))))

    return groups


def check_flatness(mol3d, threshold_flatness=0.25):
    try:
        Chem.SanitizeMol(mol3d)
        groups = get_flat_groups(mol3d)
    except Exception as e:
        return {"success": False, "error": f"flatness setup failed: {e}"}

    max_distances = []
    records = []

    for group_type, atom_indices in groups:
        coords = coords_of_atoms(mol3d, atom_indices)
        max_d = max_distance_to_plane(coords)
        passed = max_d <= threshold_flatness

        records.append({
            "type": group_type,
            "atom_indices": atom_indices,
            "max_distance": max_d,
            "pass": passed,
        })
        max_distances.append(max_d)

    return {
        "success": True,
        "num_systems_checked": len(groups),
        "num_systems_passed": sum(r["pass"] for r in records),
        "max_flatness_distance": max(max_distances) if max_distances else np.nan,
        "flatness_pass": all(r["pass"] for r in records) if records else True,
        "details": records,
        "error": None,
    }


# =========================
# 4. Full assessment for one SMILES
# =========================

def assess_one_smiles_3d(smiles, n_confs=50, num_threads=0):
    conf_res = cal_e(
        smiles,
        n_confs=n_confs,
        num_threads=num_threads,
    )

    if not conf_res["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": conf_res["error"],
        }

    mol3d = conf_res["mol"]

    geo = check_geometry(mol3d)
    flat = check_flatness(mol3d)

    if not geo["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": geo["error"],
        }

    if not flat["success"]:
        return {
            "smiles": smiles,
            "success": False,
            "three_d_pass": False,
            "error": flat["error"],
        }

    three_d_pass = (
        geo["bond_lengths_pass"]
        and geo["bond_angles_pass"]
        and geo["internal_clash_pass"]
        and flat["flatness_pass"]
    )

    return {
        "smiles": smiles,
        "success": True,
        "three_d_pass": three_d_pass,

        "energy": conf_res["energy"],
        
        "n_confs_generated": conf_res["n_confs_generated"],
        "n_confs_converged": conf_res["n_confs_converged"],

        "bond_lengths_pass": geo["bond_lengths_pass"],
        "bond_angles_pass": geo["bond_angles_pass"],
        "internal_clash_pass": geo["internal_clash_pass"],
        "flatness_pass": flat["flatness_pass"],

        "number_short_bonds": geo["number_short_bonds"],
        "number_long_bonds": geo["number_long_bonds"],
        "number_bad_angles": geo["number_bad_angles"],
        "number_clashes": geo["number_clashes"],
        "num_flat_systems_checked": flat["num_systems_checked"],
        "num_flat_systems_passed": flat["num_systems_passed"],
        "max_flatness_distance": flat["max_flatness_distance"],

        "mol3d": mol3d,
        "error": None,
    }


# =========================
# 5. Batch assessment
# =========================
import time
import multiprocessing as mp
import pandas as pd

from collections import deque
from tqdm.auto import tqdm


TIMEOUT_PER_MOLECULE = 1500


def _run_isolated_3d_worker(task, send_conn):
    """Run one SMILES assessment in an isolated process."""
    idx, smi, _, _ = task

    try:
        row = _assess_one_smiles_3d_worker(task)
        send_conn.send(row)

    except Exception as e:
        send_conn.send({
            "index": idx,
            "smiles": smi,
            "success": False,
            "three_d_pass": False,
            "is_unreasonable_3d": True,
            "failure_stage": "worker exception",
            "error": str(e),
        })

    finally:
        send_conn.close()


def assess_smiles_list_3d_parallel(
    smiles_list,
    n_confs=50,
    n_workers=32,
    num_threads_per_mol=1,
    save_all_csv_path=None,
    save_bad_csv_path=None,
    print_bad=True,
):
    """
    Parallel version of assess_smiles_list_3d.

    Each molecule is processed in an isolated process.
    A molecule exceeding 600 seconds is terminated and marked unreasonable.
    """

    tasks = deque([
        (idx, smi, n_confs, num_threads_per_mol)
        for idx, smi in enumerate(smiles_list)
    ])

    rows = [None] * len(smiles_list)
    bad_smiles = []
    bad_rows = []
    bad_count = 0

    
    start_method = (
        "fork"
        if "fork" in mp.get_all_start_methods()
        else "spawn"
    )
    ctx = mp.get_context(start_method)

    # index -> process information
    active = {}

    def launch_new_tasks():
        """Fill available worker slots."""
        while tasks and len(active) < n_workers:
            task = tasks.popleft()
            idx, smi, _, _ = task

            recv_conn, send_conn = ctx.Pipe(duplex=False)

            process = ctx.Process(
                target=_run_isolated_3d_worker,
                args=(task, send_conn),
            )
            process.start()

            # Parent only receives
            send_conn.close()

            active[idx] = {
                "process": process,
                "recv_conn": recv_conn,
                "start_time": time.monotonic(),
                "smiles": smi,
            }

    def terminate_process(process):
        """Terminate a worker process safely."""
        if process.is_alive():
            process.terminate()
            process.join(timeout=5)

        if process.is_alive():
            process.kill()
            process.join(timeout=5)

    def register_result(row):
        """Record one completed, failed, or timed-out molecule."""
        nonlocal bad_count

        idx = row["index"]
        rows[idx] = row

        if row.get("is_unreasonable_3d", False):
            bad_count += 1
            bad_smiles.append(row["smiles"])
            bad_rows.append(row)

            if print_bad:
                print(
                    f"[UNREASONABLE_3D] index={idx} | "
                    f"stage={row.get('failure_stage')} | "
                    f"error={row.get('error')} | "
                    f"SMILES:{row.get('smiles')} | "
                    f"count={bad_count}",
                    flush=True,
                )

    launch_new_tasks()

    try:
        with tqdm(total=len(smiles_list)) as progress:

            while active or tasks:
                finished_indices = []
                current_time = time.monotonic()

                for idx, info in list(active.items()):
                    process = info["process"]
                    recv_conn = info["recv_conn"]
                    elapsed = current_time - info["start_time"]

                    row = None

                    # 1. Worker returned a result
                    if recv_conn.poll():
                        try:
                            row = recv_conn.recv()
                        except EOFError:
                            row = {
                                "index": idx,
                                "smiles": info["smiles"],
                                "success": False,
                                "three_d_pass": False,
                                "is_unreasonable_3d": True,
                                "failure_stage": "worker process exited",
                                "error": (
                                    "Worker exited without returning a result"
                                ),
                            }

                        process.join(timeout=2)

                        if process.is_alive():
                            terminate_process(process)

                    # 2. Hard timeout
                    elif elapsed >= TIMEOUT_PER_MOLECULE:
                        terminate_process(process)

                        row = {
                            "index": idx,
                            "smiles": info["smiles"],
                            "success": False,
                            "three_d_pass": False,
                            "is_unreasonable_3d": True,
                            "failure_stage": "timeout",
                            "error": (
                                f"Processing exceeded "
                                f"{TIMEOUT_PER_MOLECULE} seconds"
                            ),
                        }

                    # 3. Worker crashed unexpectedly
                    elif not process.is_alive():
                        process.join()

                        # Check once more for a buffered result
                        if recv_conn.poll(0.1):
                            try:
                                row = recv_conn.recv()
                            except EOFError:
                                row = None

                        if row is None:
                            row = {
                                "index": idx,
                                "smiles": info["smiles"],
                                "success": False,
                                "three_d_pass": False,
                                "is_unreasonable_3d": True,
                                "failure_stage": "worker process exited",
                                "error": (
                                    f"Worker exited with code "
                                    f"{process.exitcode}"
                                ),
                            }

                    if row is not None:
                        recv_conn.close()
                        register_result(row)
                        finished_indices.append(idx)
                        progress.update(1)

                for idx in finished_indices:
                    active.pop(idx, None)

                launch_new_tasks()
                time.sleep(0.2)

    except KeyboardInterrupt:
        print("\nStopping all workers...", flush=True)

        for info in active.values():
            terminate_process(info["process"])
            info["recv_conn"].close()

        raise

    rows = [r for r in rows if r is not None]

    df_all = pd.DataFrame(rows)
    df_bad = pd.DataFrame(bad_rows)

    if save_all_csv_path is not None:
        df_all.to_csv(save_all_csv_path, index=False)

    if save_bad_csv_path is not None:
        df_bad.to_csv(save_bad_csv_path, index=False)

    return {
        "df_all": df_all,
        "df_bad": df_bad,
        "bad_smiles": bad_smiles,
    }

In [ ]:
emigen_smiles = read_smiles_from_txt("emigen_mols.txt")

In [ ]:

smi_list = assess_smiles_list_3d_parallel(
    emigen_smiles,
    n_confs=50,
    n_workers=16,
    num_threads_per_mol=8,
    print_bad=True,
)

bad_smiles = smi_list["bad_smiles"]

print("Total:", len(emigen_smiles))
print("Unreasonable:", len(bad_smiles))